In [1]:
!pip install fastdtw pandas numpy scipy scikit-learn


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import warnings

warnings.filterwarnings('ignore')

**1. TIỀN XỬ LÝ (LỌC NHIỄU, TÁI TẠO, VÀ CHUẨN HÓA 2D)**

In [3]:
# [NOISE ISOLATION] - Nâng ngưỡng lên 0.015 để chặn đứng Video 6
def is_dancing(df_student, variance_threshold=0.015):
    key_joints = ['left_wrist', 'right_wrist', 'left_ankle', 'right_ankle']
    df_key = df_student[df_student['Joint_Name'].isin(key_joints)]
    movement_std = df_key.groupby('Joint_Name')[['X', 'Y']].std().mean().mean()
    return movement_std >= variance_threshold, movement_std

# [SIGNAL RECONSTRUCTION & ANATOMICAL SCALING]
def preprocess_data(df):
    df = df.sort_values(by=['Frame', 'Joint_Name']).reset_index(drop=True)
    
    mask = df['Visibility'] < 0.5
    df.loc[mask, ['X', 'Y', 'Z']] = np.nan
    df[['X', 'Y', 'Z']] = df.groupby('Joint_Name')[['X', 'Y', 'Z']].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )
    df.fillna(0, inplace=True)
    
    normalized_frames = []
    for frame_id, frame_data in df.groupby('Frame'):
        l_hip = frame_data[frame_data['Joint_Name'] == 'left_hip']
        r_hip = frame_data[frame_data['Joint_Name'] == 'right_hip']
        l_sh = frame_data[frame_data['Joint_Name'] == 'left_shoulder']
        r_sh = frame_data[frame_data['Joint_Name'] == 'right_shoulder']
        
        if not l_hip.empty and not r_hip.empty and not l_sh.empty and not r_sh.empty:
            root_x = (l_hip['X'].values[0] + r_hip['X'].values[0]) / 2
            root_y = (l_hip['Y'].values[0] + r_hip['Y'].values[0]) / 2
            
            sh_x = (l_sh['X'].values[0] + r_sh['X'].values[0]) / 2
            sh_y = (l_sh['Y'].values[0] + r_sh['Y'].values[0]) / 2
            
            # Chỉ dùng 2D (X, Y) để đo Torso, loại bỏ nhiễu Z
            torso_length = np.sqrt((sh_x - root_x)**2 + (sh_y - root_y)**2)
            torso_length = max(torso_length, 1e-5)
        else:
            root_x, root_y = 0, 0
            torso_length = 1.0
            
        frame_data = frame_data.copy()
        frame_data['X_norm'] = (frame_data['X'] - root_x) / torso_length
        frame_data['Y_norm'] = (frame_data['Y'] - root_y) / torso_length
        frame_data['Z_norm'] = frame_data['Z'] # Bỏ qua Z, không dùng tính toán chuẩn
        
        normalized_frames.append(frame_data)
        
    return pd.concat(normalized_frames, ignore_index=True)

**2. XỬ LÝ ĐẶC TRƯNG (LOẠI BỎ NHIỄU TRỤC Z)**

In [4]:
def extract_joint_features(df_student, df_teacher):
    def get_vectors(df):
        # Chỉ lấy X và Y vào DTW để tăng độ mượt mà đồng bộ
        wide = df.pivot(index='Frame', columns='Joint_Name', values=['X_norm', 'Y_norm']).fillna(0)
        return wide.values, wide.index.tolist()

    vec_s, frames_s = get_vectors(df_student)
    vec_t, frames_t = get_vectors(df_teacher)
    
    distance, path = fastdtw(vec_s, vec_t, dist=euclidean)
    aligned_pairs = [{'Frame_s': frames_s[s], 'Frame_t': frames_t[t]} for s, t in path]
    df_path = pd.DataFrame(aligned_pairs)
    
    df_s_aligned = pd.merge(df_path, df_student, left_on='Frame_s', right_on='Frame').drop(columns=['Frame'])
    df_t_aligned = pd.merge(df_path, df_teacher, left_on='Frame_t', right_on='Frame').drop(columns=['Frame'])
    
    df_s_aligned = df_s_aligned.rename(columns={'X_norm': 'X_s', 'Y_norm': 'Y_s'})
    df_t_aligned = df_t_aligned.rename(columns={'X_norm': 'X_t', 'Y_norm': 'Y_t'})
    
    label_col = 'Label' if 'Label' in df_s_aligned.columns else ('Label_s' if 'Label_s' in df_s_aligned.columns else None)
    cols_to_merge_s = ['Frame_s', 'Frame_t', 'Joint_Name', 'X_s', 'Y_s'] + ([label_col] if label_col else [])
    
    df_merged = pd.merge(df_s_aligned[cols_to_merge_s], 
                         df_t_aligned[['Frame_s', 'Frame_t', 'Joint_Name', 'X_t', 'Y_t']], 
                         on=['Frame_s', 'Frame_t', 'Joint_Name'])
    
    # Tính Error và Cosine trên không gian 2D (X, Y) để khử nhiễu Camera
    df_merged['Joint_Error'] = np.sqrt((df_merged['X_s'] - df_merged['X_t'])**2 + 
                                       (df_merged['Y_s'] - df_merged['Y_t'])**2)
                                       
    dot = (df_merged['X_s']*df_merged['X_t']) + (df_merged['Y_s']*df_merged['Y_t'])
    norm_s = np.sqrt(df_merged['X_s']**2 + df_merged['Y_s']**2)
    norm_t = np.sqrt(df_merged['X_t']**2 + df_merged['Y_t']**2)
    df_merged['Joint_Cosine'] = dot / (norm_s * norm_t + 1e-9)
    
    if label_col:
        df_merged.rename(columns={label_col: 'Ground_Truth_Label'}, inplace=True)
        
    return df_merged[['Frame_s', 'Joint_Name', 'Joint_Error', 'Joint_Cosine', 'Ground_Truth_Label']]

**3. TRÍCH XUẤT DATASET TỪ CÁC VIDEO**

In [5]:
dance_pairs = [
    {"id": 1, "student": "student_dance_1.csv", "teacher": "teacher_dance_1.csv"},
    {"id": 2, "student": "student_dance_2.csv", "teacher": "teacher_dance_2.csv"},
    {"id": 3, "student": "student_dance_3.csv", "teacher": "teacher_dance_3.csv"},
    {"id": 4, "student": "student_dance_4.csv", "teacher": "teacher_dance_4.csv"},
    {"id": 5, "student": "student_dance_5.csv", "teacher": "teacher_dance_5.csv"},
    {"id": 6, "student": "student_dance_6.csv", "teacher": "teacher_dance_6.csv"},
    {"id": 7, "student": "student_dance_7.csv", "teacher": "teacher_dance_7.csv"},
    {"id": 8, "student": "student_dance_8.csv", "teacher": "teacher_dance_8.csv"},
    {"id": 9, "student": "student_dance_9.csv", "teacher": "teacher_dance_9.csv"},
    {"id": 10, "student": "student_dance_10.csv", "teacher": "teacher_dance_10.csv"},
    {"id": 11, "student": "student_dance_11.csv", "teacher": "teacher_dance_11.csv"},
    {"id": 12, "student": "student_dance_12.csv", "teacher": "teacher_dance_12.csv"},
    {"id": 13, "student": "teacher_dance_12.csv", "teacher": "teacher_dance_12.csv"},
   # {"id": 14, "student": "student_dance_14.csv", "teacher": "teacher_dance_14.csv"},
    {"id": 15, "student": "student_dance_15.csv", "teacher": "teacher_dance_15.csv"}
]

TEST_VIDEO_IDS = [5,9] 
all_joints_data = []

print("=== BẮT ĐẦU TRÍCH XUẤT CẤP ĐỘ KHỚP ===")
for pair in dance_pairs:
    try:
        df_s = pd.read_csv(pair["student"])
        df_t = pd.read_csv(pair["teacher"])
        
        is_moving, std = is_dancing(df_s)
        if not is_moving:
            print(f"[BỘ LỌC] Bỏ qua Video {pair['id']}: Đứng yên (Std={std:.4f})")
            continue
            
        print(f"[V] Đang xử lý Video {pair['id']}...")
        df_s_clean = preprocess_data(df_s)
        df_t_clean = preprocess_data(df_t)
        
        joint_features_df = extract_joint_features(df_s_clean, df_t_clean)
        joint_features_df['Video_ID'] = pair['id']
        all_joints_data.append(joint_features_df)
        
    except Exception as e:
        print(f"[-] Lỗi tại Video {pair['id']}: {e}")

master_dataset = pd.concat(all_joints_data, ignore_index=True)

=== BẮT ĐẦU TRÍCH XUẤT CẤP ĐỘ KHỚP ===
[V] Đang xử lý Video 1...
[V] Đang xử lý Video 2...
[V] Đang xử lý Video 3...
[V] Đang xử lý Video 4...
[V] Đang xử lý Video 5...
[V] Đang xử lý Video 6...
[V] Đang xử lý Video 7...
[V] Đang xử lý Video 8...
[V] Đang xử lý Video 9...
[V] Đang xử lý Video 10...
[V] Đang xử lý Video 11...
[V] Đang xử lý Video 12...
[V] Đang xử lý Video 13...
[V] Đang xử lý Video 15...


**4. CLASSIFICATION MODEL (AI) & RULE-BASED HEURISTICS (ĐÁNH GIÁ)**

In [29]:
train_data = master_dataset[~master_dataset['Video_ID'].isin(TEST_VIDEO_IDS)]
test_data = master_dataset[master_dataset['Video_ID'].isin(TEST_VIDEO_IDS)]

n_train_frames = train_data[['Video_ID', 'Frame_s']].drop_duplicates().shape[0]
n_test_frames = test_data[['Video_ID', 'Frame_s']].drop_duplicates().shape[0]

print("======================================================")
print("             THỐNG KÊ TẬP DỮ LIỆU                     ")
print("======================================================")
print(f"- Số lượng Frame tập TRAIN  : {n_train_frames:,} frames")
print(f"- Số lượng Frame tập TEST   : {n_test_frames:,} frames")
print("======================================================\n")

X_train = train_data[['Joint_Error', 'Joint_Cosine']]
y_train = train_data['Ground_Truth_Label']
X_test = test_data[['Joint_Error', 'Joint_Cosine']]

# [PHASE 3: CLASSIFICATION MODEL (AI Chấm Khớp)]
print("Đang huấn luyện AI chấm khớp (Random Forest)...")
# Đã loại bỏ class_weight='balanced' để mô hình không bị trầm cảm, đánh rớt vô tội vạ
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

test_data = test_data.copy()
test_data['Predicted_Joint_Label'] = rf_model.predict(X_test)

importances = rf_model.feature_importances_
print(f"\nĐộ quan trọng Đặc trưng: Error ({importances[0]*100:.1f}%), Cosine ({importances[1]*100:.1f}%)")

# [PHASE 4: RULE-BASED HEURISTICS (Giám Khảo Khung Hình)]
weights = {
    'left_elbow': 1.0, 'right_elbow': 1.0, 'left_knee': 1.0, 'right_knee': 1.0,       # Động tác chính (Quan trọng nhất)
        'left_wrist': 0.75, 'right_wrist': 0.75, 'left_ankle': 0.75, 'right_ankle': 0.75, # Chi tiết ngọn
        'left_shoulder': 0.5, 'right_shoulder': 0.5, 'left_hip': 0.5, 'right_hip': 0.5,   # Trụ cơ thể
        'nose': 0.2  
}



def evaluate_frame_performance(group_df, label_col):
    total_score = 0.0
    for _, row in group_df.iterrows():
        j_name = row['Joint_Name']
        if j_name in weights:
            total_score += row[label_col] * weights[j_name]
    frame_threshold = 4.7
    return 1 if total_score >= frame_threshold else 0

frame_y_true, frame_y_pred = [], []

for (video_id, frame_id), frame_df in test_data.groupby(['Video_ID', 'Frame_s']):
    frame_y_true.append(evaluate_frame_performance(frame_df, 'Ground_Truth_Label'))
    frame_y_pred.append(evaluate_frame_performance(frame_df, 'Predicted_Joint_Label'))

print("\n======================================================")
print(f"   KẾT QUẢ TỔNG HỢP (TEST VIDEO {TEST_VIDEO_IDS})")
print("======================================================\n")
print(classification_report(frame_y_true, frame_y_pred, target_names=['Fail (0)', 'Pass (1)']))

acc = accuracy_score(frame_y_true, frame_y_pred)
print(f"-> ĐỘ CHÍNH XÁC (Accuracy): {acc*100:.2f}%")

cm = confusion_matrix(frame_y_true, frame_y_pred)
if cm.shape == (2,2):
    print(f"\nCHI TIẾT TRÊN {len(frame_y_true)} FRAMES TEST:")
    print(f"- Số Frames FAIL thực, máy chấm chuẩn FAIL : {cm[0][0]}")
    print(f"- Số Frames FAIL thực, máy cho lọt PASS  : {cm[0][1]} (False Positives)")
    print(f"- Số Frames PASS thực, máy đánh oan FAIL : {cm[1][0]} (False Negatives)")
    print(f"- Số Frames PASS thực, máy chấm chuẩn PASS: {cm[1][1]}")


             THỐNG KÊ TẬP DỮ LIỆU                     
- Số lượng Frame tập TRAIN  : 6,533 frames
- Số lượng Frame tập TEST   : 613 frames

Đang huấn luyện AI chấm khớp (Random Forest)...

Độ quan trọng Đặc trưng: Error (54.1%), Cosine (45.9%)

   KẾT QUẢ TỔNG HỢP (TEST VIDEO [5, 9])

              precision    recall  f1-score   support

    Fail (0)       0.88      0.80      0.84       227
    Pass (1)       0.89      0.94      0.91       386

    accuracy                           0.89       613
   macro avg       0.89      0.87      0.88       613
weighted avg       0.89      0.89      0.89       613

-> ĐỘ CHÍNH XÁC (Accuracy): 88.74%

CHI TIẾT TRÊN 613 FRAMES TEST:
- Số Frames FAIL thực, máy chấm chuẩn FAIL : 182
- Số Frames FAIL thực, máy cho lọt PASS  : 45 (False Positives)
- Số Frames PASS thực, máy đánh oan FAIL : 24 (False Negatives)
- Số Frames PASS thực, máy chấm chuẩn PASS: 362


In [28]:
import numpy as np
from sklearn.metrics import accuracy_score, recall_score, f1_score

# 1. BAREM GIẢI PHẪU CHUẨN HÓA [0, 1] (Max = 9.25 điểm)

anatomical_weights = {
    'left_elbow': 1.0, 'right_elbow': 1.0, 'left_knee': 1.0, 'right_knee': 1.0,       # Động tác chính (Quan trọng nhất)
    'left_wrist': 0.75, 'right_wrist': 0.75, 'left_ankle': 0.75, 'right_ankle': 0.75, # Chi tiết ngọn
    'left_shoulder': 0.5, 'right_shoulder': 0.5, 'left_hip': 0.5, 'right_hip': 0.5,   # Trụ cơ thể
    'nose': 0.2                                                                      # Hướng mặt
}

print("=== GRID SEARCH: TÌM NGƯỠNG ĐIỂM (THRESHOLD) TỐI ƯU NHẤT ===")
print(f"{'Threshold':<10} | {'Accuracy':<10} | {'Recall FAIL (0)':<16} | {'Recall PASS (1)':<16} | {'F1-Macro':<10}")
print("-" * 75)

# 4.6 - 5.3
for thresh in np.arange(4.6, 5.25, 0.01):
    y_true_list = []
    y_pred_list = []
    
    for (video_id, frame_id), frame_df in test_data.groupby(['Video_ID', 'Frame_s']):
        # Lấy nhãn đúng thực tế của Frame
        true_label = frame_df['Ground_Truth_Label'].mode()[0] 
        
        # Tính tổng điểm theo Barem
        total_score = sum(row['Predicted_Joint_Label'] * anatomical_weights.get(row['Joint_Name'], 0) for _, row in frame_df.iterrows())
        
        # Chốt nhãn dự đoán
        pred_label = 1 if total_score >= thresh else 0
        
        y_true_list.append(true_label)
        y_pred_list.append(pred_label)
        
    acc = accuracy_score(y_true_list, y_pred_list)
    rec_0 = recall_score(y_true_list, y_pred_list, pos_label=0)
    rec_1 = recall_score(y_true_list, y_pred_list, pos_label=1)
    f1 = f1_score(y_true_list, y_pred_list, average='macro')
    
    # In ra báo cáo để tìm điểm cân bằng
    print(f"{thresh:<10.2f} | {acc*100:<9.2f}% | {rec_0*100:<15.2f}% | {rec_1*100:<15.2f}% | {f1:.4f}")

=== GRID SEARCH: TÌM NGƯỠNG ĐIỂM (THRESHOLD) TỐI ƯU NHẤT ===
Threshold  | Accuracy   | Recall FAIL (0)  | Recall PASS (1)  | F1-Macro  
---------------------------------------------------------------------------
4.60       | 92.01    % | 88.67          % | 93.66          % | 0.9101
4.61       | 92.01    % | 88.67          % | 93.66          % | 0.9101
4.62       | 92.01    % | 88.67          % | 93.66          % | 0.9101
4.63       | 92.01    % | 88.67          % | 93.66          % | 0.9101
4.64       | 92.01    % | 88.67          % | 93.66          % | 0.9101
4.65       | 92.01    % | 88.67          % | 93.66          % | 0.9101
4.66       | 92.01    % | 88.67          % | 93.66          % | 0.9101
4.67       | 92.01    % | 88.67          % | 93.66          % | 0.9101
4.68       | 92.01    % | 88.67          % | 93.66          % | 0.9101
4.69       | 92.01    % | 88.67          % | 93.66          % | 0.9101
4.70       | 92.01    % | 88.67          % | 93.66          % | 0.9101
4.71   

In [15]:
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np

print("=== TÌM TRỌNG SỐ KHÁCH QUAN BẰNG LOGISTIC REGRESSION ===")

# 1. Dùng mô hình Random Forest dự đoán trên chính tập Train để làm dữ liệu đầu vào
train_data_copy = train_data.copy()
train_data_copy['Pred_Joint'] = rf_model.predict(X_train)

# 2. Xoay dữ liệu (Pivot): Mỗi Frame là 1 dòng, 13 khớp là 13 cột (chứa giá trị 0 hoặc 1)
frame_labels = []
frame_features = []

for (vid, fid), group in train_data_copy.groupby(['Video_ID', 'Frame_s']):
    # Lấy nhãn đúng của toàn bộ Frame
    true_label = group['Ground_Truth_Label'].mode()[0]
    
    # Tạo vector 13 chiều chứa kết quả Đúng/Sai của từng khớp
    joint_preds = group.set_index('Joint_Name')['Pred_Joint'].to_dict()
    
    frame_labels.append(true_label)
    frame_features.append(joint_preds)

# Chuyển thành DataFrame ma trận
X_frame_train = pd.DataFrame(frame_features).fillna(0)
y_frame_train = frame_labels

# 3. Huấn luyện Logistic Regression để tìm độ tương quan tuyến tính
# Không dùng fit_intercept để ép trọng số đi qua gốc tọa độ
lr_model = LogisticRegression(fit_intercept=False, random_state=42)
lr_model.fit(X_frame_train, y_frame_train)

# 4. Trích xuất Trọng số (Coefficients) và Chuẩn hóa về thang Max = 1.0
raw_weights = lr_model.coef_[0]
raw_weights = np.maximum(0, raw_weights) # Cắt bỏ các trọng số âm (nếu có)
max_weight = max(raw_weights)

if max_weight > 0:
    learned_weights = {joint: round(w / max_weight, 2) for joint, w in zip(X_frame_train.columns, raw_weights)}
else:
    learned_weights = {joint: 0 for joint in X_frame_train.columns}

print("\n[BẢNG TRỌNG SỐ DO AI TỰ HỌC TỪ DỮ LIỆU THỰC TẾ]")
print("Cậu hãy copy Dictionary này dán đè vào biến 'weights' ở code cũ:\n")

print("weights = {")
items = list(sorted(learned_weights.items(), key=lambda item: item[1], reverse=True))
for i in range(0, len(items), 4):
    chunk = items[i:i+4]
    line = "    " + ", ".join([f"'{k}': {v:.2f}" for k, v in chunk]) + ","
    print(line)
print("}")

=== TÌM TRỌNG SỐ KHÁCH QUAN BẰNG LOGISTIC REGRESSION ===

[BẢNG TRỌNG SỐ DO AI TỰ HỌC TỪ DỮ LIỆU THỰC TẾ]
Cậu hãy copy Dictionary này dán đè vào biến 'weights' ở code cũ:

weights = {
    'right_wrist': 1.00, 'left_wrist': 0.84, 'right_ankle': 0.57, 'left_elbow': 0.25,
    'left_knee': 0.13, 'right_knee': 0.11, 'left_ankle': 0.10, 'left_hip': 0.00,
    'left_shoulder': 0.00, 'nose': 0.00, 'right_elbow': 0.00, 'right_hip': 0.00,
    'right_shoulder': 0.00,
}
